<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/001.%20Agentic%20Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Dive Agentic Retrieval Augmented Generation

An Agentic RAG is required when we use reasoning to determine which action(s) to take and in which order to take them. Essentially we use agents instead of a LLM directly to accomplish a set of tasks which requires planning, multi step reasoning, tool use and/or learning over time. Agents give us agency!

Agency : The ability to take action or to choose what action to take

In the context of RAG, we can plug in agents to enhance the reasoning prior to selection of RAG pipelines, within a RAG pipeline for retrieval or reranking and finally for synthesising before we send out the response. This improves RAG to a large extent by automating complex workflows and decisions that are required for a non trivial RAG use case.

### Purpose of this Agentic RAG
This notebook presents a practical implementation of Agentic Retrieval-Augmented Generation (RAG)—a system where decision-making and tool selection are delegated to an intelligent agent before executing a response. Rather than passing every query through a static RAG pipeline, this system introduces agency—the ability to choose the best course of action depending on the nature of the query.

At the heart of this implementation is a router prompt, which classifies user queries into one of three categories:

- OpenAI documentation: Queries related to tools, APIs, or usage guidelines for OpenAI models
- 10-K financial reports: Questions requiring retrieval from company filings or financial datasets
- Live Internet search: Broader, current, or comparative queries that need web access

Once the query is classified, the system invokes a corresponding route handler:

- For OpenAI and 10-K queries, it retrieves relevant context from a vector database (Qdrant) using text embeddings, then applies a RAG-based response generator.
- For Internet queries, it fetches real-time information using a web-access API (ARES).

This approach is an example of Agentic RAG, where reasoning precedes retrieval and generation. By plugging in agents before and within the RAG pipeline, we make the system smarter and more adaptive. This allows us to:

- Automatically choose the right retrieval method based on context
- Combine structured knowledge with real-time search
- Scale RAG beyond trivial use cases by integrating multi-step decision logic

Importantly, no external agentic frameworks are used—this is a ground-up implementation that demonstrates how to build a lightweight but intelligent agentic system using only a language model, prompt engineering, and retrieval tools.

## Setup and Dependencies

In [1]:
# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install transformers==4.48.0
!pip install tavily-python

In [2]:
# Import basic libraries
import requests             # Used for making HTTP requests (e.g., calling ARES API for live internet queries)
import json                 # For parsing and structuring JSON data (especially OpenAI and routing responses)

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower()) or os.getenv(
                name.replace("TAVILY_API_KEY", "TAVILYAPI_KEY"))

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library to interface with GPT models for routing and generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Optional visualization (for analysis/debugging purposes)
import matplotlib.pyplot as plt       # For displaying charts or visual debug outputs (e.g., embeddings visualizations)
import matplotlib.image as mpimg      # For loading/displaying images if needed (rare in RAG, but helpful in demos)

# Embedding models (used for text vectorization during retrieval)
from transformers import AutoTokenizer, AutoModel  # For loading custom transformer models if not using OpenAI embeddings

from qdrant_client import models

import qdrant_client
import asyncio
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops
# # Vector database client
# from qdrant_client import QdrantClient   # Qdrant is used as the vector store to retrieve documents based on similarity

## 1. Defining the Internet Tool

First, we will define a tool function that enables our system to answer queries requiring real-time, internet-based information. Not all questions can be answered using static documents like OpenAI docs or financial filings—sometimes users ask about current trends, comparisons, or live updates.

To handle this, we introduce a live search capability using the **Tavily**.

### What is Tavily?  
It is a specialized search engine and API built specifically for AI agents and large language models (LLMs).
Unlike traditional search engines designed for humans (like Google or Bing), Tavily cleans, filters, and structures web data so AI models can process and reason over it without hallucinating or wasting token space

Please generate the API key [here](https://www.tavily.com/)


In [3]:
#loads Tavily api key from colab secrets
tavily_api_key=userdata.get('TAVILY_API_KEY')

In [4]:
#import requests  # For sending HTTP requests to the Tavily Api
from tavily import TavilyClient
#from tavily.errors import TavilyError

# Initialize client
tavily_client = TavilyClient(api_key=tavily_api_key)

def get_internet_content(user_query: str, action: str):
    """
    Fetches a response from the internet using Tavily based on the user's query.

    This function serves as the tool invoked when the router classifies a query
    as requiring real-time information beyond internal datasets—i.e., "INTERNET_QUERY".
    It sends the query to Tavily and returns structured results.

    Args:
        user_query (str): The user's question that needs a live answer.
        action (str): Route type (always expected to be "INTERNET_QUERY").

    Returns:
        str: Response text from live Google search results or an error message.
    """
    print("Getting your response from the internet 🌐 ...")

    try:
        # Search using Tavily (include_answer generates a synthesized summary)
        response = tavily_client.search(
            query=user_query,
            max_results=5,
            include_answer=True
        )

        parts = []

        # Tavily's direct synthesized answer (if available)
        if response.get("answer"):
            parts.append(f"[Direct Answer] {response['answer']}")

        # Top search results: titles, content snippets, and URLs
        for i, result in enumerate(response.get("results", []), start=1):
            title = result.get("title", "")
            content = result.get("content", "")
            url = result.get("url", "")
            if content:
                parts.append(f"[{i}] {title}\n    {content}\n    Source: {url}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    #except TavilyError as tavily_err:
      #return f"Tavily API error occurred: {tavily_err}"
    except Exception as err:
      return f"An error occurred while fetching search results: {err}"



In [5]:
print(get_internet_content("Tell me about best travel destinations in 2026?","INTERNET_QUERY")) #run internet function to test results

Getting your response from the internet 🌐 ...
[Direct Answer] In 2026, top travel destinations include Cyprus, Slovenia, and Ireland, known for their unique cultural experiences and natural beauty. Dominica and Queenstown are emerging for sustainable tourism. Popular activities include adventure and eco-friendly travel.

[1] My Top 26 Best Travel Destinations For 2026
    ## Cyprus

The island of Cyprus is one of the largest in the Mediterranean. It’s also one of the most beautiful. With Turkish and Greek influence, Cyprus has the best of both worlds. It’s a pretty overlooked travel destination, but one of the most rewarding for those who do make it here.

## Jordan [...] Guatemala is a paradise for adventurers, with the grueling hike up to Acatenango being one of my top bucket list experiences I’ve ever done.

## Albania [...] While Kenya and Tanzania are most travelers’ go-to destinations of East Africa, Uganda is not to be overlooked. I initially ended up in Uganda out of pure curio

## 2. Router Query Function — Giving the Agent Its Brain

In this step, we will define the router function, which plays a critical role in our Agentic RAG system.

### What is a Router?

A router is like the decision-making brain of our assistant.

Before trying to answer a user's question, the system first needs to figure out:

> “Where should I go to find the right answer?”

To make this decision, we use the OpenAI GPT model. We provide it with a detailed system prompt that explains how to classify the user's question into one of these categories:

- **OPENAI_QUERY** → Questions about OpenAI tools, APIs, models, or documentation.
- **10K_DOCUMENT_QUERY** → Questions about companies, financial filings, or analysis based on 10-K reports.
- **INTERNET_QUERY** → Anything else that likely requires real-time or general web information.

### What does the function do?

- Sends the user's question to the OpenAI API.
- Receives a JSON response containing:
  - `action`: The category the query belongs to.
  - `reason`: A short explanation for the decision.
  - `answer`: (Optional) A quick response if it’s simple enough (left blank for internet queries).
- Parses the response and returns it as a Python dictionary.

### Why is this important?

This router gives the system agency—the ability to decide which knowledge source to use. It’s what makes this pipeline agentic, not just static.

Without the router, every query would follow the same path. With it, we can:

- Dynamically switch between tools and data sources.
- Handle different types of user questions intelligently.
- Avoid wasting resources on unnecessary steps.


## Query Routing Workflow

The diagram below shows the full decision flow — from receiving a user query to returning a final response.

```
                        ┌─────────────────────┐
                        │     User Query      │
                        └──────────┬──────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │      Router LLM (GPT-4o)     │
                    │         route_query()         │
                    │                              │
                    │  Reads the query and decides │
                    │  which data source to use    │
                    └──────────────┬───────────────┘
                                   │
           ┌───────────────────────┼───────────────────────┐
           │                       │                       │
           ▼                       ▼                       ▼
┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│    OPENAI_QUERY     │ │ 10K_DOCUMENT_QUERY  │ │   INTERNET_QUERY    │
│                     │ │                     │ │                     │
│ e.g. "What are      │ │ e.g. "What was      │ │ e.g. "Best LLMs     │
│  OpenAI Agents?"    │ │  Uber's revenue?"   │ │  in 2026?"          │
└──────────┬──────────┘ └──────────┬──────────┘ └──────────┬──────────┘
           │                       │                        │
           ▼                       ▼                        ▼
┌─────────────────────┐ ┌─────────────────────┐  ┌──────────────────────┐
│   Embed Query       │ │   Embed Query        │  │      TavilyApi         │
│  (Nomic Model)      │ │  (Nomic Model)       │  │  get_internet_       │
│                     │ │                      │  │  content()           │
│  get_text_          │ │  get_text_           │  │                      │
│  embeddings()       │ │  embeddings()        │  │  Live Google search  │
└──────────┬──────────┘ └──────────┬──────────┘  └──────────┬───────────┘
           │                       │                          │
           ▼                       ▼                          │
┌─────────────────────┐ ┌─────────────────────┐              │
│  Qdrant Vector DB   │ │  Qdrant Vector DB   │              │
│  Collection:        │ │  Collection:         │              │
│  "opnai_data"       │ │  "10k_data"          │              │
│                     │ │                      │              │
│  Retrieve top-3     │ │  Retrieve top-3      │              │
│  similar chunks     │ │  similar chunks      │              │
└──────────┬──────────┘ └──────────┬──────────┘              │
           │                       │                          │
           └───────────┬───────────┘                          │
                       ▼                                      │
           ┌───────────────────────┐                          │
           │    RAG Response       │                          │
           │    Generator          │                          │
           │  rag_formatted_       │                          │
           │  response()           │                          │
           │                       │                          │
           │  GPT-4 synthesizes    │                          │
           │  answer from context  │                          │
           │  + adds citations     │                          │
           └───────────┬───────────┘                          │
                       │                                      │
                       └──────────────────┬───────────────────┘
                                          ▼
                             ┌────────────────────────┐
                             │     Final Response     │
                             │       to User          │
                             └────────────────────────┘
```

### Key decision points at a glance

| Route | Trigger | Retrieval Method | Response Generator |
|---|---|---|---|
| `OPENAI_QUERY` | OpenAI docs, APIs, Agents | Qdrant `opnai_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `10K_DOCUMENT_QUERY` | Financial filings, company revenue | Qdrant `10k_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `INTERNET_QUERY` | Anything else / real-time info | Tavily Api live Google search | Raw search result snippets |

> **Note:** Both vector-based routes share the same embedding model (`nomic-embed-text-v1.5`) and RAG generator — only the Qdrant collection changes. The router's JSON output (`action` field) is the single decision variable that drives the entire flow.


In [6]:
# Securely retrieve the OpenAI API key from Colab's user data store
# This avoids hardcoding sensitive credentials directly in the notebook
openai_api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client with the retrieved API key
# This client will be used for:
# - Query classification via the router prompt
# - Potentially generating responses from retrieved context
openaiclient = OpenAI(api_key=openai_api_key)


In [7]:
from openai import OpenAIError

def route_query(user_query: str):
    router_system_prompt =f"""
    As a professional query router, your objective is to correctly classify user input into one of three categories based on the source most relevant for answering the query:
    1. "OPENAI_QUERY": If the user's query appears to be answerable using information from OpenAI's official documentation about Agents, tools, models, APIs, or services (e.g., guardrails, agents, what is an agent, embeddings, moderation API, usage guidelines).
    2. "10K_DOCUMENT_QUERY": If the user's query pertains to a collection of documents from the 10k annual reports, datasets, or other structured documents, typically for research, analysis, or financial content.
    3. "INTERNET_QUERY": If the query is neither related to OpenAI nor the 10k documents specifically, or if the information might require a broader search (e.g., news, trends, tools outside these platforms), route it here.

    Your decision should be made by assessing the domain of the query.

    Always respond in this valid JSON format:
    {{
        "action": "OPENAI_QUERY" or "10K_DOCUMENT_QUERY" or "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words answer. Leave empty if INTERNET_QUERY"
    }}

    EXAMPLES:

    - User: "How to fine-tune GPT-3?"
    Response:
    {{
        "action": "OPENAI_QUERY",
        "reason": "Fine-tuning is OpenAI-specific",
        "answer": "Use fine-tuning API"
    }}

    - User: "Where can I find the latest financial reports for the last 10 years?"
    Response:
    {{
        "action": "10K_DOCUMENT_QUERY",
        "reason": "Query related to annual reports",
        "answer": "Access through document database"
    }}

    - User: "Top leadership styles in 2024"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Needs current leadership trends",
        "answer": ""
    }}

    - User: "What's the difference between ChatGPT and Claude?"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Cross-comparison of different providers",
        "answer": ""
    }}

    Strictly follow this format for every query, and never deviate.
    User: {user_query}
    """

    try:
        # Query the GPT-4 model with the router prompt and user input
        response = openaiclient.chat.completions.create(
            model="gpt-5.6-luna",
            messages=[{"role": "system", "content": router_system_prompt}]
        )

        # Extract and parse the model's JSON response
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        json_text = json_match.group()
        parsed_response = json.loads(json_text)
        return parsed_response

    # Handle OpenAI API errors (e.g., rate limits, authentication)
    except OpenAIError as api_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"OpenAI API error: {api_err}",
            "answer": ""
        }

    # Handle case where model response isn't valid JSON
    except json.JSONDecodeError as json_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"JSON parsing error: {json_err}",
            "answer": ""
        }

    # Catch-all for any other unforeseen issues
    except Exception as err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"Unexpected error: {err}",
            "answer": ""
        }

In [8]:
route_query("what is the revenue of uber in 2021?")


{'action': '10K_DOCUMENT_QUERY',
 'reason': "Requests Uber's annual financial revenue",
 'answer': "Uber's 2021 revenue"}

In [9]:
route_query("what is an AI Agent?")

{'action': 'OPENAI_QUERY',
 'reason': 'AI Agents are covered in OpenAI documentation',
 'answer': 'Software that performs tasks'}

## 3. Setting Up Qdrant Vector Database for Agentic RAG
In this step, we are connecting our agent to a pre-built vector database using Qdrant—a tool used to store and search document embeddings (numerical representations of text).

What Are We Doing?
We are loading an existing Qdrant database that was downloaded from a GitHub repository. This database already contains:

- Vectorized OpenAI documentation
- Vectorized 10-K financial filings

By loading this saved data:

- We save time (no need to re-embed the documents)
- We enable fast similarity search to retrieve relevant text chunks

This setup allows our system to perform semantic search, meaning it can understand the meaning of the user query and match it with the most relevant pieces of information stored in the database.


### Why This Matters in Agentic RAG
Once the router decides that the query should go to the OpenAI docs or the 10-K reports, our system uses Qdrant to:

- Search for the most relevant pieces of text
- Pass those to the model to generate a grounded answer

So, this step is essential to support retrieval-augmented generation (RAG) within our agentic flow.

#Data Sources:

**10K Database: Lyft 2024 & Uber 2021 SEC filings**

**OpenAI Docs: Official OpenAI documentation about Agents**

For lecture demo purposes, the vecitr database has already been created and hosted on Github which we will clone here. In order to create your own embeddings, the notebook and data will be hosted and shared on github

In [10]:
# Colab only — wipe a previous clone if you need a clean copy
if IN_COLAB:
    !rm -rf /content/multi-agent-course

In [11]:
# The prebuilt Qdrant collections (10-K + OpenAI docs) ship with the repo.
# On Colab we clone to get them; locally you already have them.
if IN_COLAB:
    !git clone https://github.com/hamzafarooq/multi-agent-course.git

Cloning into 'multi-agent-course'...
remote: Enumerating objects: 2912, done.
remote: Counting objects: 100% (523/523), done.
remote: Compressing objects: 100% (285/285), done.
remote: Total 2912 (delta 282), reused 328 (delta 231), pack-reused 2389 (from 2)
Receiving objects: 100% (2912/2912), 124.97 MiB | 16.76 MiB/s, done.
Resolving deltas: 100% (1096/1096), done.
Updating files: 100% (1027/1027), done.


In [12]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# Prebuilt collections (10-K and OpenAI docs) — cloned on Colab, already present locally.
import os

_MODULE = "modules/Module_3_Production_Agentic_RAG_AI_Systems"
if IN_COLAB:
    QDRANT_PATH = f"/content/multi-agent-course/{_MODULE}/Agentic_RAG/qdrant_data"
else:
    # the notebook lives in the module folder, so the data sits right next to it
    QDRANT_PATH = os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data")

print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: /content/multi-agent-course/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data


## 4. Building the Retriever and RAG for Vector Databases
In this section, we build the core logic that allows our agent to find relevant documents and generate grounded answers using them.

###Step 1: Import the Embedding Model
We start by importing the nomic-ai/nomic-embed-text-v1.5 model from Hugging Face. This model is used to convert any text (such as a user query) into a dense vector, known as an embedding. These embeddings capture the semantic meaning of text, allowing us to later compare and retrieve similar documents.


In [13]:
# Load the tokenizer and embedding model from Hugging Face
# This model converts raw text into dense vector representations (embeddings)
# Used for similarity search in Qdrant during document retrieval
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

def get_text_embeddings(text):
    """
    Converts input text into a dense embedding using the Nomic embedding model.
    These embeddings are used to query Qdrant for semantically relevant document chunks.

    Args:
        text (str): The input text or query from the user.

    Returns:
        np.ndarray: A fixed-size vector representing the semantic meaning of the input.
    """
    # Tokenize and prepare input for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Forward pass to get model outputs
    outputs = text_model(**inputs)

    # Take the mean across all token embeddings to get a single vector (pooled representation)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert to NumPy array and detach from computation graph
    return embeddings[0].detach().numpy()

# Example usage: Generate and preview the embedding of a test sentence
text = "This is a test sentence."
embeddings = get_text_embeddings(text)
print(embeddings[:5])  # Print first 5 dimensions for inspection


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[ 1.2799693   0.40158385 -3.5162656  -0.39813167  1.5919145 ]


### Step 2: Define the Embedding Function
We then define a function get_text_embeddings() which:

- Tokenizes the input text
- Runs it through the model
- Computes the average of all token embeddings
- Returns a single vector that represents the full sentence

This vector will be used to query Qdrant to find the most relevant document chunks based on similarity.

In [14]:
def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with article references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from retrieved document chunks (context). It prompts the model to generate a
    grounded response, explicitly citing sources using a reference format.

    Args:
        user_query (str): The user's original question.
        context (list): List of text chunks retrieved from Qdrant (10-K or OpenAI docs).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting context documents
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on chunk IDs or order.
    rag_prompt = f"""
       Based on the given context, answer the user query: {user_query}\nContext:\n{context}
       and employ references to the ID of articles provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc. </instructions>
    """

    #  Call GPT-5.6-Luna to generate the response using the RAG-style prompt
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content


### Step 3: Define the RAG Response Generator
After retrieving relevant text chunks from Qdrant, we use the rag_formatted_response() function to generate a final answer. This function:

- Takes the user query and the retrieved document chunks
- Builds a prompt that asks the language model (GPT-5.6-Luna) to answer the question using only the provided context
- Instructs the model to include references like [1], [2] for traceability

This ensures the output is not only informative but also grounded in actual retrieved data.

Together, these two functions lay the foundation for combining retrieval (from vector DB) and generation (from LLM) — the two pillars of a RAG system.



In [15]:
async def retrieve_and_response(user_query: str, action: str):
    """
    Retrieves relevant text chunks from the appropriate Qdrant collection
    based on the query type, then generates a response using RAG.

    This function powers the retrieval and response generation pipeline
    for queries that are classified as either OPENAI-related or 10-K related.
    It uses semantic search to fetch relevant context from a Qdrant vector store
    and then generates a response using that context via a RAG prompt.

    Args:
        user_query (str): The user's input question.
        action (str): The classification label from the router (e.g., "OPENAI_QUERY", "10K_DOCUMENT_QUERY").

    Returns:
        str: A model-generated response grounded in retrieved documents, or an error message.
    """

    # Define mapping of routing labels to their respective Qdrant collections
    collections = {
        "OPENAI_QUERY": "opnai_data",           # Collection of OpenAI documentation embeddings
        "10K_DOCUMENT_QUERY": "10k_data"        # Collection of 10-K financial document embeddings
    }

    try:
        # Ensure that the provided action is valid
        if action not in collections:
            return "Invalid action type for retrieval."

        # Step 1: Convert the user query into a dense vector (embedding)
        try:
            query = get_text_embeddings(user_query)
        except Exception as embed_err:
            return f"Embedding error: {embed_err}"  # Fail early if embedding fails

        # Step 2: Retrieve top-matching chunks from the relevant Qdrant collection
        try:
            text_hits = await client.query_points(
                collection_name=collections[action],  # Choose the right collection based on routing
                query=query,                          # The embedding of the user's query
                limit=3                               # Fetch top 3 relevant chunks
            )
        except Exception as qdrant_err:
            return f"Vector DB query error: {qdrant_err}"  # Handle Qdrant access issues

        # Extract the raw content from the retrieved vector hits
        contents = [point.payload['content'] for point in text_hits.points]

        # If no relevant content is found, return early
        if not contents:
            return "No relevant content found in the database."

        # Step 3: Pass the retrieved context to the RAG model to generate a response
        try:
            response = rag_formatted_response(user_query, contents)
            return response
        except Exception as rag_err:
            return f"RAG response error: {rag_err}"  # Handle generation failures

    # Catch any unforeseen errors in the overall process
    except Exception as err:
        return f"Unexpected error: {err}"


# 5. Putting It All Together: Running the Agentic RAG
In this final step, we combine everything into a single function that controls the entire Agentic RAG workflow. The agentic_rag() function acts as the main orchestrator of the system.

Here’s what it does:

- Prints the user's query for reference.
- Uses the router function (powered by GPT) to decide which type of data source to use:
  - OpenAI documentation
  - 10-K financial reports
- Internet search
- Calls the correct function based on the route:
- If it’s an OpenAI or 10-K query, it retrieves data from Qdrant and generates a RAG response.
- If it’s an Internet query, it uses the ARES API to fetch live information.
- Displays the final response, neatly formatted in the console.

This step brings the agentic loop full circle—from understanding the question, reasoning about where to search, to finally responding with the best possible answer.

In [16]:
# Dictionary that maps the route labels (decided by the router) to their respective functions
# Each type of query is handled differently:
# - OPENAI_QUERY and 10K_DOCUMENT_QUERY use document retrieval + RAG
# - INTERNET_QUERY uses a web search API
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}

def agentic_rag(user_query: str):
    """
    Main function that runs the full Agentic RAG system.

    This function takes a user's question, decides what type of query it is (OpenAI-related,
    financial document-related, or general internet), and then calls the right function
    to handle it. Finally, it prints out the full conversation and response.

    Args:
        user_query (str): The user's input question.

    Returns:
        None (It just prints the result nicely to the console)
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    try:
        # Step 1: Print the user's original question to the console
        print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

        # Step 2: Use the router (powered by GPT) to decide which route the query belongs to
        try:
            response = route_query(user_query)
        except Exception as route_err:
            # If something goes wrong while classifying the query, show an error message
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"Routing error: {route_err}\n")
            return

        # Extract the routing decision and the reason behind it
        action = response.get("action")  # e.g., "OPENAI_QUERY"
        reason = response.get("reason")  # e.g., "Related to OpenAI tools"

        # Step 3: Show the selected route and why it was chosen
        print(f"{GREY}📍 Selected Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        # Step 4: Call the correct function depending on the route (retrieval or web search)
        try:
            route_function = routes.get(action)  # Find the function to use for this route
            if route_function:
                if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                    # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                    result = asyncio.run(route_function(user_query, action))
                else:
                    # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                    result = route_function(user_query, action)
            else:
                result = f"Unsupported action: {action}"  # Catch unknown routing types
        except Exception as exec_err:
            result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

        # Step 5: Print the final response to the user
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")

    except Exception as err:
        # Catch-all for any unexpected errors in the overall logic
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"Unexpected error occurred: {err}\n")


In [17]:
agentic_rag("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Requests Uber's annual financial revenue
⚙️ Processing query...

🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



In [18]:
agentic_rag("what was lyft revenue in 2022?")

👤 User Query: what was lyft revenue in 2022?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Query asks for Lyft's annual financial revenue
⚙️ Processing query...

🤖 BOT RESPONSE:

Lyft’s revenue in 2022 was **$4.095 billion** (approximately **$4,095,135,000**). [1]



In [19]:
agentic_rag("List me down new LLMs in 2025")

👤 User Query: List me down new LLMs in 2025

📍 Selected Route: INTERNET_QUERY
📝 Reason: Requires current information about newly released LLMs
⚙️ Processing query...

Getting your response from the internet 🌐 ...
🤖 BOT RESPONSE:

[Direct Answer] In 2025, notable large language models included DeepSeek-V3-0324, GPT-4.5, and Gemini 2.5 Pro. Mistral Small 3.2 and Devstral 2 were also released. Granite 4 introduced a new hybrid architecture.

[1] A list of large language models (LLMs)
    Mistral Small 3.2 is a 24B LLM released in June 2025. Its performance is comparable to that of the more recent Ministral 3 14B.
 Devstral is Mistral’s agentic engineering-focused model series. Devstral 2, released in December 2025, comprises two models. Devstral 2 123B is released under a modified MIT License, requiring organizations with over $20M USD in monthly revenue to request a commercial license from Mistral. Devstral Small 2 24B is released under standard Apache 2.0 license. [...] Granite 4, launc

In [20]:
agentic_rag("how to work with chat completions?")

👤 User Query: how to work with chat completions?

📍 Selected Route: OPENAI_QUERY
📝 Reason: Chat Completions is an OpenAI API
⚙️ Processing query...

🤖 BOT RESPONSE:

## Working with Chat Completions

Chat Completions lets you send a sequence of messages to a model and receive an assistant response. The conversation is represented with roles such as `system`, `user`, and `assistant`.

### 1. Install and initialize the SDK

```bash
pip install openai
```

```python
from openai import OpenAI

client = OpenAI()  # Reads OPENAI_API_KEY from the environment
```

### 2. Send a basic request

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Explain how chat completions work."
        }
    ]
)

answer = response.choices[0].message.content
print(answer)
```

The main components are:

- `

In [21]:
agentic_rag("best ways to build Agents")

👤 User Query: best ways to build Agents

📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI agent documentation
⚙️ Processing query...

🤖 BOT RESPONSE:

## Best ways to build agents

1. **Choose an appropriate use case**  
   Build an agent when the task involves a multi-step workflow and requires the system to make decisions, use tools, adapt to changing information, or act on the user’s behalf. Simple chatbots, single-turn generation, and classifiers generally do not need agents. [1]

2. **Start with a simple architecture**  
   An agent fundamentally needs three components:

   - **Model:** Handles reasoning and workflow decisions  
   - **Tools:** APIs or functions used to retrieve information or take actions  
   - **Instructions:** Define behavior, policies, and constraints [1]

3. **Prototype with the most capable model**  
   Use a strong model initially to establish a performance baseline. Once the workflow performs reliably, test smaller and faster mod

## 6. Role-Based Access Control (RBAC)

Everything so far assumes one kind of user: whoever asks gets whatever the router
picks. In a real deployment that's rarely true. An engineer shouldn't be able to pull
finance's 10-K numbers out of the vector store, and a finance analyst has no business
reading internal engineering docs — even though both are talking to the same agent.

RBAC puts a **permission check between the router's decision and the tool call**. The
router still reasons about *where* the answer lives; RBAC decides whether *this user*
is allowed to go there. If not, the request is rejected before any embedding, vector
search, or grounding call happens.

**This demo — 2 roles, 3 knowledge sources:**

| Knowledge source | Route label | `engineer` | `finance_analyst` |
|---|---|---|---|
| 📘 OpenAI documentation (Qdrant) | `OPENAI_QUERY` | ✅ | ✅ |
| 📗 10-K filings (Qdrant) | `10K_DOCUMENT_QUERY` | ❌ | ✅ |
| 🌐 Live internet search (TavilyApi) | `INTERNET_QUERY` | ✅ | ❌ |

The two Qdrant collections and the TavilyApi tool are the same ones built above — RBAC
is a layer on top, not a different pipeline.


In [22]:
# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider. In production this comes from SSO/JWT
# claims or an internal users table — never a dict in the notebook.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

# ── Roles → the route labels each role may reach ─────────────────────────────
# This is an allow-list: anything not listed here is denied by default.
ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

# Human-readable names, used only for clearer denial messages
SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    """True only if this user's role is explicitly allowed to use this route."""
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    """Every route label this user may reach — useful for constraining the router."""
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())


print("alice  (engineer)        →", allowed_sources("alice"))
print("bob    (finance_analyst) →", allowed_sources("bob"))
print("carol  (unknown user)    →", allowed_sources("carol"))

alice  (engineer)        → {'OPENAI_QUERY', 'INTERNET_QUERY'}
bob    (finance_analyst) → {'OPENAI_QUERY', '10K_DOCUMENT_QUERY'}
carol  (unknown user)    → set()


In [23]:
def secure_agentic_rag(user_id: str, user_query: str):
    """
    The same agentic RAG loop as above, with one addition: after the router picks a
    route, the user's role must permit that route before the tool is called.

    Order of operations:
        1. Identify the user  → unknown users are rejected immediately
        2. Route the query    → router decides which knowledge source fits
        3. RBAC check         → role allowed to use that source? deny if not
        4. Retrieve + answer  → only ever reached by an authorized request

    Args:
        user_id (str): Who is asking (looked up in USERS).
        user_query (str): The question.

    Returns:
        str: The answer, or a denial message.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )

    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    # Step 1 — unknown identity is denied before anything else runs
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.\n")
        return f"🚫 Access denied: unknown user '{user_id}'."

    # Step 2 — the router still does the reasoning about where the answer lives
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        return f"Routing error: {route_err}"

    action = decision.get("action")
    reason = decision.get("reason")
    print(f"{GREY}📍 Selected Route: {action}")
    print(f"📝 Reason: {reason}{RESET}\n")

    # Step 3 — the gate. Nothing is embedded, searched, or grounded past this point
    #          unless the role is permitted to use the chosen source.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source}.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {source}."
        )

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")

    # Step 4 — identical to agentic_rag() from Section 5
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(user_query, action))
        else:
            result = route_function(user_query, action)
    except Exception as exec_err:
        result = f"Execution error: {exec_err}"

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result

### Demo — same question, different roles

Each pair below sends the *identical* query as `alice` (engineer) and `bob`
(finance_analyst). The router makes the same decision both times — only the
permission check differs.


In [24]:
print("=" * 70)
print("1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("alice", "what was uber revenue in 2021?")

1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks about Uber's annual financial revenue

🚫 ACCESS DENIED — role 'engineer' may not query 10-K financial filings.



"🚫 Access denied: your role ('engineer') does not have permission to query 10-K financial filings."

In [25]:
print("=" * 70)
print("2) bob (finance_analyst) asks the same question → ALLOWED")
print("=" * 70)
secure_agentic_rag("bob", "what was uber revenue in 2021?")

2) bob (finance_analyst) asks the same question → ALLOWED
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for a company's annual financial revenue

✅ Access granted — processing...

🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]'

In [26]:
print("=" * 70)
print("3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("bob", "List me down new LLMs in 2025")

3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
❓ Query: List me down new LLMs in 2025

📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests current information about newly released LLMs

🚫 ACCESS DENIED — role 'finance_analyst' may not query live internet search.



"🚫 Access denied: your role ('finance_analyst') does not have permission to query live internet search."

In [27]:
print("=" * 70)
print("4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED")
print("=" * 70)
secure_agentic_rag("alice", "best ways to build Agents")

4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED
👤 User: alice  (role: engineer)
❓ Query: best ways to build Agents

📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI agent documentation

✅ Access granted — processing...

🤖 BOT RESPONSE:

## Best ways to build reliable agents

### 1. Start with the right use case
Build an agent when the task involves a multi-step workflow that requires decisions, tool use, and some degree of autonomy—such as resolving support issues, booking reservations, changing code, or generating reports. A simple chatbot, classifier, or single-turn LLM call usually does not need an agent.[1]

Before building, define:

- The user’s goal
- The workflow steps required
- Which decisions the agent must make
- When the task is complete
- What should happen if the agent fails or lacks information

### 2. Use the three core components

A practical agent consists of:

1. **Model** – handles reasoning and decision-making  
2. **Tools** 

'## Best ways to build reliable agents\n\n### 1. Start with the right use case\nBuild an agent when the task involves a multi-step workflow that requires decisions, tool use, and some degree of autonomy—such as resolving support issues, booking reservations, changing code, or generating reports. A simple chatbot, classifier, or single-turn LLM call usually does not need an agent.[1]\n\nBefore building, define:\n\n- The user’s goal\n- The workflow steps required\n- Which decisions the agent must make\n- When the task is complete\n- What should happen if the agent fails or lacks information\n\n### 2. Use the three core components\n\nA practical agent consists of:\n\n1. **Model** – handles reasoning and decision-making  \n2. **Tools** – APIs or functions used to retrieve information or take actions  \n3. **Instructions** – define behavior, workflow rules, and guardrails[2]\n\nFor example, a support agent might use:\n\n- An LLM for deciding what to do\n- A customer database tool for retrie

**Where this is still weak — and how you'd harden it:**

- **The router runs before the check.** One LLM call is spent classifying a query the
  user may not be allowed to ask. That's cheap and leaks nothing, but you can do
  better: pass `allowed_sources(user_id)` into the router prompt so it only ever
  chooses from routes the role can reach, and deny anything that falls outside.
- **This gates whole sources, not chunks.** When one collection mixes content that
  different roles may only *partially* see, push the check into the vector store with
  **payload-based filters** (Qdrant supports this natively) so restricted chunks never
  enter the retrieved context in the first place. You'll do exactly this, at file
  granularity, in `003. Agentic Router_semantic_caching_rbac.ipynb`.
- **Caching and RBAC interact badly if you're careless.** A shared semantic cache
  keyed only on the question will happily serve `bob`'s finance answer to `alice`.
  Any cache sitting behind an access check must be partitioned by role (or by the
  permitted source set) — think about this before you add one.
- **Every check is an audit point.** Log who asked for what and whether it was
  allowed; that trail is what makes the system defensible in a regulated environment.


**Required:** Part 1 — sub-query division.

**Bonus (optional):** RBAC with a semantic cache. It extends Section 6 and is a
useful warm-up for **ARGUS**, where you build multi-source retrieval with a real caching
layer and have to report cost with and without the cache.

| | Task | Status | Builds on |
|---|---|---|---|
| **Part 1** | Sub-query division | **Required** | Sections 2 & 5 |
| **Bonus** | RBAC + semantic cache, without cross-role leakage | Optional | Section 6 |


---

## Part 1 — Sub-query division

Right now a compound question is treated as one search. Ask *"What was Uber's revenue
in 2021 and what was Lyft's in 2024?"* and the router picks a single route and fires a
single retrieval — so you get a partial answer, or a muddled one.

Next part: break compound queries into focused sub-queries, run each one through the
full agentic pipeline independently, then compose a single coherent answer.

**Requirements**

1. Write `agentic_rag_multi(user_query)` that:
   - calls `sub_queries()` to split the query (reference implementation below),
   - **routes each sub-query separately** — they may legitimately land on different
     sources (one on `10K_DOCUMENT_QUERY`, another on `INTERNET_QUERY`),
   - collects the per-sub-query answers and synthesises **one** final response,
   - preserves citations from each sub-answer in the composed output.
2. Handle the single-question case without regression — one question in, one route,
   no extra LLM calls beyond the split.
3. Parse the model's JSON defensively. `sub_queries()` returns a *string*; it can come
   back wrapped in prose or a code fence. Don't let a malformed split crash the agent —
   fall back to treating the input as one query.

**Check against these**

| Query | Expected behaviour |
|---|---|
| `"what was uber revenue in 2021?"` | 1 sub-query, 1 route, same as `agentic_rag()` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 sub-queries, **different** routes |

**Stretch:** run the sub-queries concurrently with `asyncio.gather` instead of in
sequence, and compare wall-clock time.


In [28]:
import json
import re
import asyncio

# Standardized ANSI color codes for terminal logging
CYAN = "\033[96m"
GREY = "\033[90m"
BOLD = "\033[1m"
RESET = "\033[0m"

def sub_queries(user_query: str) -> str:
    """
    Asks the LLM to analyze a query and break it down into atomic sub-queries if needed.
    Returns a raw string (which may contain JSON wrapped in Markdown code blocks or prose).
    """
    prompt = f"""
    You are an expert query decomposition assistant.
    Analyze the following user query. If it contains multiple sub-questions or requests distinct pieces of information
    that could come from different sources (e.g., company 10-Ks, OpenAI docs, or the internet), split it into separate sub-queries.
    If it is a single question, return a list containing only that single question.

    Strictly return your answer as a JSON array of strings, like this:
    ["sub-query 1", "sub-query 2"]

    User Query: {user_query}
    """
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[{"role": "system", "content": prompt}]
    )
    return response.choices[0].message.content


In [29]:
def parse_sub_queries_defensively(raw_response: str, fallback_query: str) -> list[str]:
    """
    Defensively parses JSON arrays from LLM output string, handling code fences,
    surrounding prose, or completely malformed outputs.
    """
    if not raw_response or not isinstance(raw_response, str):
        return [fallback_query]

    # Clean code fences if present
    cleaned = re.sub(r"```(?:json)?", "", raw_response).strip("` \n\r\t")

    # Extract JSON array using regular expressions
    match = re.search(r"\[.*\]", cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)

    # Attempt JSON parse
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, list) and len(parsed) > 0:
            return [str(q).strip() for q in parsed if str(q).strip()]
    except Exception:
        pass

    # Fallback to treating input as one query if split fails or is invalid
    return [fallback_query]

In [30]:
async def process_single_sub_query(sub_query: str) -> dict:
    """
    Routes and executes a single sub-query concurrently, returning its action, sub-query, and raw answer.
    """
    # Step 1: Route sub-query
    route_info = route_query(sub_query)
    action = route_info.get("action", "INTERNET_QUERY")

    # Step 2: Dispatch to the appropriate handler
    route_function = routes.get(action, get_internet_content)

    if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
        answer = await route_function(sub_query, action)
    else:
        answer = route_function(sub_query, action)

    return {
        "sub_query": sub_query,
        "action": action,
        "answer": answer
    }

In [31]:
def process_section_with_citations(sub_query: str, raw_answer: str, action: str) -> str:
    """
    Passes a single sub-query result through the LLM to clean up the prose for its dedicated section,
    explicitly instructing it to maintain every inline citation and source link.
    """
    prompt = f"""
    You are a technical editor refining a specific section of a multi-part answer.

    Sub-Query Focus: "{sub_query}" [Source: {action}]
    Raw Section Content:
    {raw_answer}

    Task:
    - Clean up, format, and summarize the content to make it clear and direct.
    - CRITICAL: You MUST keep ALL inline citations, source links, brackets, and references (e.g., [1], [Source], URLs) exactly as they appear in the raw content. Do not drop or rewrite any citation.
    """

    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[{"role": "system", "content": prompt}]
    )
    content = response.choices[0].message.content.strip()
    # Strip any redundant top-level markdown headers generated inside the section
    content = re.sub(r"^#{1,3}\s+.*?\n+", "", content).strip()
    return content

In [32]:
def compose_section_based_response(user_query: str, sub_results: list[dict]) -> str:
    """
    Appends processed sub-answers section by section under structural markdown headers,
    guaranteeing that citations remain anchored to their specific sub-query context.
    """
    # 1. Format context from all executed sub-queries
    context_summary = "\n".join([f"- {res['sub_query']}: {res['answer'][:300]}..." for res in sub_results])

    # 2. Supply the context directly to the intro prompt for brief high-level summary paragraph
    intro_prompt = f"""
    Provide a 1-to-2 sentence direct overview answering the user query at a high level.
    Base your summary ONLY on the sub-query context provided below.
    Do not dive into deep details, as detailed sections follow.

    User Query: {user_query}

    Sub-Query Context:
    {context_summary}
    """

    intro_response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[{"role": "system", "content": intro_prompt}]
    )

    output_parts = [intro_response.choices[0].message.content.strip(), "\n---"]

    # Append each sub-query response under its own section header
    for idx, res in enumerate(sub_results, 1):
        refined_content = process_section_with_citations(
            res["sub_query"],
            res["answer"],
            res["action"]
        )

        section_md = f"### {idx}. {res['sub_query'].rstrip('?')}\n**Source Type:** `{res['action']}`\n\n{refined_content}"
        output_parts.append(section_md)

    return "\n\n".join(output_parts)

In [33]:
print(sub_queries("what was lyft revenue in 2021 and what was uber revenue in 2021"))

["What was Lyft's revenue in 2021?", "What was Uber's revenue in 2021?"]


In [34]:

def agentic_rag_multi(user_query: str) -> str:
    """
    Multi-query Agentic RAG driver function:
    1. Splits complex queries defensively.
    2. Runs sub-queries concurrently via asyncio.gather.
    3. Assembles outputs using Section-Based Appending to preserve exact citations.
    """
    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

    # Step 1: Sub-query generation and defensive parsing
    raw_split = sub_queries(user_query)
    queries = parse_sub_queries_defensively(raw_split, user_query)

    # Fast path for single-query execution
    if len(queries) <= 1:
        single_q = queries[0] if queries else user_query
        route_info = route_query(single_q)
        action = route_info.get("action", "INTERNET_QUERY")
        reason = route_info.get("reason", "")

        print(f"{GREY}📍 Selected Single Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        route_function = routes.get(action, get_internet_content)
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(single_q, action))
        else:
            result = route_function(single_q, action)

        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")
        return result

    # Step 2: Display sub-query execution plan
    print(f"{GREY}🔀 Split into {len(queries)} sub-queries:{RESET}")
    for idx, q in enumerate(queries, 1):
        print(f"{GREY}  {idx}. {q}{RESET}")
    print()

    # Step 3: Run sub-queries concurrently using asyncio.gather
    async def _run_all():
        tasks = [process_single_sub_query(q) for q in queries]
        return await asyncio.gather(*tasks)

    sub_results = asyncio.run(_run_all())

    for res in sub_results:
        print(f"{GREY}📍 Sub-Query: \"{res['sub_query']}\" -> Route: {res['action']}{RESET}")
    print(f"\n{GREY}⚙️ Assembling response with section-based appending...{RESET}\n")

    # Step 4: Compose multi-section response
    final_response = compose_section_based_response(user_query, sub_results)

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{final_response}\n")
    return final_response


# Test cases from the table above
# agentic_rag_multi("what was uber revenue in 2021?")
# agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")
# agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

In [35]:
agentic_rag_multi("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?

📍 Selected Single Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks about a company's annual financial revenue
⚙️ Processing query...

🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**), up 57% from 2020. [1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**), up 57% from 2020. [1]'

In [36]:
agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")

👤 User Query: what was lyft revenue in 2021 and what was uber revenue in 2021

🔀 Split into 2 sub-queries:
  1. What was Lyft's revenue in 2021?
  2. What was Uber's revenue in 2021?

📍 Sub-Query: "What was Lyft's revenue in 2021?" -> Route: 10K_DOCUMENT_QUERY
📍 Sub-Query: "What was Uber's revenue in 2021?" -> Route: 10K_DOCUMENT_QUERY

⚙️ Assembling response with section-based appending...

🤖 BOT RESPONSE:

In 2021, Lyft reported revenue of **$4.095 billion**, while Uber reported revenue of **$17.455 billion**.


---

### 1. What was Lyft's revenue in 2021
**Source Type:** `10K_DOCUMENT_QUERY`

Lyft’s revenue in 2021 was **$4.095 billion** (approximately **$4,095,135,000**). [1]

### 2. What was Uber's revenue in 2021
**Source Type:** `10K_DOCUMENT_QUERY`

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



"In 2021, Lyft reported revenue of **$4.095 billion**, while Uber reported revenue of **$17.455 billion**.\n\n\n---\n\n### 1. What was Lyft's revenue in 2021\n**Source Type:** `10K_DOCUMENT_QUERY`\n\nLyft’s revenue in 2021 was **$4.095 billion** (approximately **$4,095,135,000**). [1]\n\n### 2. What was Uber's revenue in 2021\n**Source Type:** `10K_DOCUMENT_QUERY`\n\nUber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]"

In [37]:
agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

👤 User Query: what was uber's 2021 revenue and what are the newest LLMs?

🔀 Split into 2 sub-queries:
  1. What was Uber's revenue in 2021?
  2. What are the newest large language models (LLMs)?

Getting your response from the internet 🌐 ...
📍 Sub-Query: "What was Uber's revenue in 2021?" -> Route: 10K_DOCUMENT_QUERY
📍 Sub-Query: "What are the newest large language models (LLMs)?" -> Route: INTERNET_QUERY

⚙️ Assembling response with section-based appending...

🤖 BOT RESPONSE:

Uber’s 2021 revenue was **$17.455 billion** (approximately **$17.5 billion**). Among the newest LLMs are **GPT-4o** and **Claude 4 Sonnet**, offering advanced multimodal capabilities and long-context memory.


---

### 1. What was Uber's revenue in 2021
**Source Type:** `10K_DOCUMENT_QUERY`

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]

### 2. What are the newest large language models (LLMs)
**Source Type:** `INTERNET_QUERY`

[Source: INTERNET_QUERY] The newest LLMs named 

"Uber’s 2021 revenue was **$17.455 billion** (approximately **$17.5 billion**). Among the newest LLMs are **GPT-4o** and **Claude 4 Sonnet**, offering advanced multimodal capabilities and long-context memory.\n\n\n---\n\n### 1. What was Uber's revenue in 2021\n**Source Type:** `10K_DOCUMENT_QUERY`\n\nUber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]\n\n### 2. What are the newest large language models (LLMs)\n**Source Type:** `INTERNET_QUERY`\n\n[Source: INTERNET_QUERY] The newest LLMs named in the provided sources include OpenAI’s GPT-4o, Anthropic’s Claude Sonnet 4, Google’s Gemini models, Meta’s Llama models, DeepSeek’s open models, xAI’s Grok models, and Mistral’s latest open-weight models. The newest models increasingly support multimodal input, real-time voice interaction, long context windows, coding, reasoning, and tool use.\n\n### Notable recent models\n\n- **GPT-4o:** OpenAI’s multimodal flagship, released in May 2024. It supports text, image

### Stretch — sequential vs. concurrent: wall-clock time

**`asyncio.gather` alone does not make the sub-queries run at the same time**

Think of `asyncio` as **one cook with several pots**. The cook can switch between pots, but only at moments when they say *"I'll wait here"*, which is what `await` means. The first version of `process_single_sub_query()` used `gather`, but every slow step inside it (the router, the RAG answer, the Tavily search) is an ordinary blocking call. It's like the cook standing and staring at one pot until it boils: the other pots don't get touched, so the sub-queries still run **one after another**.

**The fix:** `asyncio.to_thread()` hands each blocking call to a helper (a background thread). The cook starts all the pots, and the helpers wait on them at the same time. The updated `process_single_sub_query()` above does this, and `agentic_rag_multi()` uses it automatically.

**The comparison below** times only the sub-query stage (route + retrieve + answer for every sub-query). The split and the final composition are the same in every mode, so they're left out. It runs three modes on the same sub-queries:

| Mode | What it does | Expected time |
|---|---|---|
| 1. Sequential | One sub-query at a time | ≈ the **sum** of all sub-query times |
| 2. `gather`, blocking calls | The original code | ≈ the same as sequential — `gather` alone doesn't help |
| 3. `gather` + threads | The new code | ≈ the **slowest single** sub-query |

Tavily API response times vary from call to call, so some noise is expected. Each run makes real API calls (3 modes × the number of sub-queries), so keep `runs` small.

In [38]:
import time, io, contextlib
from statistics import mean


async def _process_blocking(sub_query: str) -> dict:
    """The ORIGINAL process_single_sub_query(): async, but every slow call blocks the event loop."""
    route_info = route_query(sub_query)
    action = route_info.get("action", "INTERNET_QUERY")
    if action in ("OPENAI_QUERY", "10K_DOCUMENT_QUERY"):
        answer = await retrieve_and_response(sub_query, action)
    else:
        answer = get_internet_content(sub_query, "INTERNET_QUERY")
    return {"sub_query": sub_query, "action": action, "answer": answer}


async def _one_after_another(queries):
    return [await process_single_sub_query(q) for q in queries]


async def _all_together(process_fn, queries):
    return await asyncio.gather(*(process_fn(q) for q in queries))


def compare_wall_clock(user_query: str, runs: int = 1):
    """Time the sub-query stage of agentic_rag_multi() in three modes and print a table."""
    queries = parse_sub_queries_defensively(sub_queries(user_query), user_query)
    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}")
    print(f"{GREY}🔀 {len(queries)} sub-queries:{RESET}")
    for i, q in enumerate(queries, 1):
        print(f"{GREY}  {i}. {q}{RESET}")

    modes = {
        "1. Sequential":                    lambda: _one_after_another(queries),
        "2. gather, blocking (original)":   lambda: _all_together(_process_blocking, queries),
        "3. gather + threads (new)":        lambda: _all_together(process_single_sub_query, queries),
    }
    times = {name: [] for name in modes}

    for run in range(1, runs + 1):
        print(f"\n{GREY}Run {run}/{runs}...{RESET}")
        for name, make_coroutine in modes.items():
            start = time.perf_counter()
            with contextlib.redirect_stdout(io.StringIO()):   # hide per-call log lines
                results = asyncio.run(make_coroutine())
            times[name].append(time.perf_counter() - start)
        routes_used = ", ".join(r["action"] for r in results)
        print(f"{GREY}  routes: {routes_used}{RESET}")

    baseline = mean(times["1. Sequential"])
    print(f"\n{'Mode':<34}{'avg time':>10}{'speed-up':>10}")
    print("-" * 54)
    for name, values in times.items():
        avg = mean(values)
        print(f"{name:<34}{avg:>9.2f}s{baseline / avg:>9.2f}x")
    return times

In [39]:
# Three sub-queries on two different sources: two 10-K lookups + one internet search.
# Expect mode 2 ≈ mode 1, and mode 3 noticeably faster (close to the slowest single sub-query).
compare_wall_clock(
    "what was uber revenue in 2021, what was lyft revenue in 2024, and what are the newest LLMs?",
    runs=1,
)

👤 User Query: what was uber revenue in 2021, what was lyft revenue in 2024, and what are the newest LLMs?
🔀 3 sub-queries:
  1. What was Uber's revenue in 2021?
  2. What was Lyft's revenue in 2024?
  3. What are the newest large language models (LLMs)?

Run 1/1...
  routes: 10K_DOCUMENT_QUERY, 10K_DOCUMENT_QUERY, INTERNET_QUERY

Mode                                avg time  speed-up
------------------------------------------------------
1. Sequential                          8.90s     1.00x
2. gather, blocking (original)         9.27s     0.96x
3. gather + threads (new)             13.81s     0.64x


{'1. Sequential': [8.900962608000555],
 '2. gather, blocking (original)': [9.272027330000128],
 '3. gather + threads (new)': [13.812551879998864]}

---

## Bonus (optional) — RBAC with a semantic cache

> **Come back to this after notebooks 002 and 003** — `002. Semantic Caching.ipynb`
> for how a FAISS cache works, and `003. Agentic Router_semantic_caching_rbac.ipynb` for `SemanticCaching`
> in `rag_helpers.py` and the file-level RBAC gate you'll be extending. You can read the spec now;
> you'll have every piece you need once those two are done.

Section 6 gates access by role. A semantic cache makes repeat questions near-instant.
Put them together naively and you build a data leak: the cache is keyed on the
*question*, so once `bob` (finance_analyst) asks about Uber's revenue, `alice`
(engineer) asks the same thing, hits the cache, and is handed finance data the RBAC
gate was supposed to deny her — without a single retrieval ever running.

Your job: add caching to `secure_agentic_rag()` so that repeat questions are fast
**and** no answer ever crosses a permission boundary.

**Requirements**

1. Build a role-aware cache. Two viable designs — pick one and justify it in a comment:
   - **Partitioned:** a separate FAISS index (or a namespace) per role, so a lookup
     can only ever see entries its own role produced.
   - **Tagged:** one index, but each entry stores the role (or the permitted source
     set) that produced it, and a hit is only honoured when it matches the caller.
2. Write `secure_agentic_rag_cached(user_id, user_query)` with this order of
   operations — it matters:
   ```
   unknown user      → DENIED   (no embedding, no cache, no LLM)
   route the query   → which source does this need?
   role not allowed  → DENIED   (still no cache lookup — a denial must not be cacheable)
   cache lookup      → HIT  → return stored answer
                     → MISS → run the pipeline, store, return
   ```
3. Return a dict, not a bare string, so the self-check can verify behaviour:
   `{"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}`
4. Never cache a denial, and never cache a time-sensitive query. Reuse the
   `is_time_sensitive()` idea from `003. Agentic Router_semantic_caching_rbac.ipynb` — a stock
   price cached for an hour is a wrong answer served fast.
5. Make the leak test below pass.

**Hints**

- `SemanticCaching` in `rag_helpers.py` is a working FAISS cache — read it first.
  Partitioning it is mostly a matter of what you key and what you search.
- Think about what happens when a user's role *changes*. Should their old cache
  entries still be reachable? Write a sentence on your answer.
- Two roles share `OPENAI_QUERY`. A strictly per-role cache re-computes that answer
  once per role — correct, but wasteful. Caching per *permitted source* instead of per
  role fixes it. Trade-off worth a comment.

**Stretch:** log every request as an audit record — user, role, query, route, decision,
cache status, latency — and print a small table at the end. That table is what you'd
hand an auditor.


### Bonus — how it's built (design choices)

**Order of checks** (the same order the spec asks for):

```
unknown user      → DENIED   (no embedding, no cache, no LLM)
route the query   → which source does this need?
role not allowed  → DENIED   (no cache lookup; a denial is never cached)
time-sensitive?   → answer fresh, never read or write the cache
cache lookup      → HIT  → return the stored answer
                  → MISS → run the pipeline, store it (only if it succeeded), return
```
### **SUBMISSION NOTES**
**Design choice 1: Partitioned, not tagged**

- **Tagged** = one shared filing cabinet with a sticker on every folder saying whose it is. You find the closest folder, *then* check the sticker, and put it back if it's the wrong one. Other people's answers **are** looked at and rejected by a single check.
- **Partitioned** = separate filing cabinets. You only open the one you're allowed to open. Other answers are **never** looked at.

**partitioned** option used here because:
- FAISS can't check stickers while it searches. It returns the closest entry whatever its sticker. A rejected wrong-sticker entry can hide a valid match sitting right behind it.
- With tags, one forgotten or buggy check leaks everything. With partitions, the wrong answers simply aren't in the cabinet being searched.
- Each partition is just the ordinary `SemanticCaching` from `rag_helpers.py`, so there's no new cache code.

**Design choice 2: One partition per *source*, not per *role***

- **Per role:** a Finance cabinet and an Engineering cabinet. Both roles may use the OpenAI docs, so the same OpenAI answer gets computed and stored twice, once in each.
- **Per source:** one cabinet per knowledge source (`OPENAI_QUERY`, `10K_DOCUMENT_QUERY`, `INTERNET_QUERY`). The permission check runs **first**, and then only the approved source's cabinet is opened.

**per source** used because:
- Shared answers are computed once. If alice asks about the OpenAI docs, bob's similar question is a cache hit.
- If a role loses access to a source, it can no longer reach that source's cabinet, including old answers in it.
- A question the router sends to the wrong place is only compared with answers from that one source.

*Trade-off:* this assumes an answer depends only on the source and the question, never on who asked. That's true here. If roles ever saw different parts of one collection (e.g. Qdrant filters by company), the cabinet key would need to become "source + filter".

**Never cached:** denials, time-sensitive questions, and failed answers (errors, "no results").

**When a user's role changes:** the role is looked up fresh on every request and nothing is stored per user, so the user immediately gets exactly their new role's sources. Nothing from their old access carries over.

**Setup notes:**
- This uses the **updated** `rag_helpers.py` (the version that accepts `encoder=`). On Colab, upload it to `/content` with the Files panel.
- Don't call `init_rag()` here. This notebook already has its own Qdrant client (`client`), and a second client on the same folder fails.

In [40]:
# ── Bonus setup (1/2): libraries for the semantic cache ──────────────────────
# The cache comes from rag_helpers.py (notebook 003) and needs these libraries,
# which this notebook's first install cell doesn't include. The versions match
# notebook 003 and don't change the transformers version already installed above.
!pip install -q "sentence-transformers==3.4.1" "einops==0.8.1" faiss-cpu

In [41]:
# ── Bonus setup (2/2): the SemanticCaching class ─────────────────────────────
# Copied from rag_helpers.py (notebook 003), so no file upload is needed.
# Only what the bonus uses is kept: time-sensitivity check, lookup, storage.

# NOTE: import sentence_transformers BEFORE faiss. Both ship their own OpenMP
# runtime; on macOS, loading a SentenceTransformer after faiss kills the kernel.
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json, re, time


class SemanticCaching:
    """
    FAISS-backed semantic cache with a time-sensitivity filter.
    It only looks up and stores answers; the caller decides what to do on a hit or miss.
    """

    # Matched as WHOLE words/phrases, so "now" does not match "know".
    TIME_SENSITIVE_KEYWORDS = [
        "today", "tonight", "now", "currently", "current",
        "latest", "recent", "recently", "right now", "at the moment",
        "at present", "as of now", "this week", "this month", "this year",
        "this quarter", "this season", "this morning", "this afternoon",
        "this evening", "this weekend", "yesterday", "tomorrow",
        "last week", "last month", "last year", "upcoming", "live",
        "breaking", "just happened", "what time", "what day", "what date",
        "happening now", "events today", "news today", "news this week",
        "stock price", "stock prices", "share price", "share prices",
        "weather", "forecast", "temperature",
        "real-time", "realtime", "schedule today", "outage", "outages",
    ]

    def __init__(self, json_file="rag_cache.json", threshold=0.30, clear_on_init=False, encoder=None):
        """
        Args:
            json_file:     File used to save the cache to disk.
            threshold:     Max squared L2 distance for a hit (lower = stricter). 0.30 is the
                           value measured in notebook 002: paraphrases ~0.16–0.25, related-
                           but-different questions ~0.38+.
            clear_on_init: If True, start with an empty cache.
            encoder:       An already-loaded SentenceTransformer to share between several
                           caches. If None, the Nomic model is loaded here.
        """
        self.embedding_dim = 768
        self.index = faiss.IndexFlatL2(self.embedding_dim)
        self.euclidean_threshold = threshold
        self.json_file = json_file
        self._time_pattern = re.compile(
            r"\b(?:" + "|".join(re.escape(k) for k in self.TIME_SENSITIVE_KEYWORDS) + r")\b"
        )
        self.encoder = encoder or SentenceTransformer(
            "nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True
        )
        if clear_on_init:
            self.clear_cache()
        else:
            self.load_cache()

    def is_time_sensitive(self, question: str) -> bool:
        """True if the question contains a time word — such answers are never cached."""
        return bool(self._time_pattern.search(question.lower()))

    def clear_cache(self):
        """Empty the cache and the FAISS index, and overwrite the file."""
        self.cache = {"questions": [], "embeddings": [], "response_text": []}
        self.index = faiss.IndexFlatL2(self.embedding_dim)
        self.save_cache()

    def load_cache(self):
        """Load entries from the file and rebuild the FAISS index in the same order."""
        try:
            with open(self.json_file, "r") as f:
                self.cache = json.load(f)
            if self.cache["embeddings"]:
                self.index.add(np.array(self.cache["embeddings"], dtype=np.float32))
        except FileNotFoundError:
            self.cache = {"questions": [], "embeddings": [], "response_text": []}

    def save_cache(self):
        with open(self.json_file, "w") as f:
            json.dump(self.cache, f)

    def check_cache(self, question: str):
        """
        Embed the question and find the closest stored question.
        Returns (hit, answer, embedding, similarity, row_id). The embedding is returned
        even on a miss so it doesn't have to be computed again when storing.
        """
        embedding = self.encoder.encode([question], normalize_embeddings=True)
        if self.index.ntotal == 0:
            return False, None, embedding, None, None
        D, I = self.index.search(embedding, 1)
        distance = float(D[0][0])  # squared L2 distance
        if I[0][0] != -1 and distance <= self.euclidean_threshold:
            row_id = int(I[0][0])
            similarity = 1.0 - distance / 2.0  # cosine similarity, for normalised vectors
            return True, self.cache["response_text"][row_id], embedding, similarity, row_id
        return False, None, embedding, None, None

    def add_to_cache(self, question: str, answer: str, embedding):
        """Store a question + answer and save to disk."""
        self.cache["questions"].append(question)
        self.cache["embeddings"].append(embedding[0].tolist())
        self.cache["response_text"].append(answer)
        self.index.add(embedding)
        self.save_cache()


print("✅ SemanticCaching defined")

✅ SemanticCaching defined


In [42]:
# Reuses USERS / ROLE_PERMISSIONS / has_access / SOURCE_LABELS (Section 6),
# route_query (Section 2) and routes (Section 5).

_SHARED_ENCODER = None   # the cache's embedding model: loaded once per session, shared by all partitions


def _get_shared_encoder():
    global _SHARED_ENCODER
    if _SHARED_ENCODER is None:
        print("Loading the cache embedding model (once)...")
        _SHARED_ENCODER = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
    return _SHARED_ENCODER


class RoleAwareSemanticCache:
    """
    A semantic cache that cannot serve an answer across a permission boundary.

    Design choice: PARTITIONED, one partition per knowledge SOURCE (route label).

    Stub change: check()/add() take the SOURCE instead of user_id, because the source
    is what the cache is partitioned by. The user is checked before check() is called.
    """

    SOURCES = ("OPENAI_QUERY", "10K_DOCUMENT_QUERY", "INTERNET_QUERY")

    def __init__(self, threshold: float = 0.30):
        # 0.30 = the threshold measured in notebook 002 (also rag_helpers' default)
        encoder = _get_shared_encoder()
        self.partitions = {
            source: SemanticCaching(
                json_file=f"rbac_cache_{source}.json",
                threshold=threshold,
                clear_on_init=True,
                encoder=encoder,          # all partitions share one loaded model
            )
            for source in self.SOURCES
        }

    def is_time_sensitive(self, question: str) -> bool:
        """Same keyword check as notebook 003 ("today", "now", "latest", "stock price", ...)."""
        return self.partitions[self.SOURCES[0]].is_time_sensitive(question)

    def check(self, source: str, question: str):
        """Search ONLY this source's partition. Returns (hit, answer, embedding, similarity)."""
        hit, answer, embedding, similarity, _row_id = self.partitions[source].check_cache(question)
        return hit, answer, embedding, similarity

    def add(self, source: str, question: str, answer: str, embedding):
        """Store an answer in this source's partition."""
        self.partitions[source].add_to_cache(question, answer, embedding)

    def sizes(self) -> dict:
        """Number of cached answers per partition."""
        return {source: len(p.cache["questions"]) for source, p in self.partitions.items()}


# 001's handlers return failures as ordinary text instead of raising an error.
# These are the messages they can produce; an answer that starts with one of them
# is never cached, so a temporary failure isn't served to later questions.
_ERROR_PREFIXES = (
    "Invalid action type", "Embedding error:", "Vector DB query error:",
    "No relevant content found", "RAG response error:", "Unexpected error:",
    "No results found", "An error occurred while fetching search results:",
    "Execution error:", "Unsupported action:",
)


def _run_route(action: str, user_query: str):
    """Run the handler for one approved route (same dispatch as agentic_rag). Returns (answer, ok)."""
    try:
        route_function = routes[action]
        if action in ("OPENAI_QUERY", "10K_DOCUMENT_QUERY"):
            answer = asyncio.run(route_function(user_query, action))
        else:
            answer = route_function(user_query, action)
    except Exception as err:
        return f"Execution error: {err}", False
    ok = isinstance(answer, str) and bool(answer.strip()) and not answer.startswith(_ERROR_PREFIXES)
    return answer, ok


def secure_agentic_rag_cached(user_id: str, user_query: str, cache: RoleAwareSemanticCache) -> dict:
    """
    RBAC-gated agentic RAG with a role-aware semantic cache.

    Order: identity → route → permission → time check → cache → pipeline.

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None,
               "source": route label | None, "cached": bool}
        "cached" is True when the answer came from, or was stored in, the cache.
    """
    CYAN, GREY, RED, GREEN, YELLOW, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[93m", "\033[1m", "\033[0m"
    )
    start = time.time()
    role = USERS.get(user_id)          # looked up fresh every time (see "role changes" above)

    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    def _result(answer, status, source=None, cached=False):
        colour = {"HIT": GREEN, "MISS": YELLOW, "DENIED": RED}[status]
        print(f"{colour}{BOLD}Status: {status}{RESET}  ({time.time() - start:.2f}s)")
        if status != "DENIED":
            print(f"\n{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n{answer}\n")
        return {"answer": answer, "status": status, "role": role, "source": source, "cached": cached}

    # 1. Unknown user → denied before any embedding, cache lookup or LLM call
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.")
        return _result(f"🚫 Access denied: unknown user '{user_id}'.", "DENIED")

    # 2. Route → which source does this question need?
    decision = route_query(user_query)
    action = decision.get("action") or "UNKNOWN_ROUTE"
    print(f"{GREY}📍 Selected Route: {action}\n📝 Reason: {decision.get('reason')}{RESET}\n")

    # 3. Permission → denied BEFORE any cache lookup, and the denial is never cached
    if not has_access(user_id, action):
        source_name = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source_name}.")
        return _result(
            f"🚫 Access denied: your role ('{role}') does not have permission to query {source_name}.",
            "DENIED", action,
        )

    # 4. Time-sensitive → always answered fresh; the cache is neither read nor written
    if cache.is_time_sensitive(user_query):
        print(f"{YELLOW}⏰ Time-sensitive — skipping the cache for a fresh answer.{RESET}")
        answer, _ok = _run_route(action, user_query)
        return _result(answer, "MISS", action, cached=False)

    # 5. Cache lookup — ONLY in the partition of the route that was just approved
    hit, answer, embedding, similarity = cache.check(action, user_query)
    if hit:
        print(f"{GREEN}✅ Cache HIT{RESET} in the {action} partition (similarity {similarity:.3f})")
        return _result(answer, "HIT", action, cached=True)

    # 6. Cache miss → run the pipeline; store the answer only if it succeeded
    print(f"{YELLOW}❌ Cache MISS{RESET} — running the {action} pipeline...")
    answer, ok = _run_route(action, user_query)
    if ok:
        cache.add(action, user_query, answer, embedding)
        print(f"{GREEN}💾 Stored in the {action} partition.{RESET}")
    else:
        print(f"{RED}⚠️ The pipeline failed — the answer was NOT cached.{RESET}")
    return _result(answer, "MISS", action, cached=ok)

In [43]:
# ── Bonus: self-check — this must pass ───────────────────────────────────────
# It asserts behaviour, not wording, so your answer text can be anything.
# Steps 1–6 are the original checks. Steps 7–8 were added: 7 tests the per-source
# design, 8 tests requirement 4 (time-sensitive questions are never cached).

def _step(text):
    print("\n" + "=" * 70 + f"\n{text}\n" + "=" * 70)


def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin  = "what was uber revenue in 2021?"
    q_doc  = "how do I build an agent with the OpenAI Agents SDK?"
    q_live = "What are the latest AI tools this week?"

    _step("1) bob asks a finance question for the first time → MISS")
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    _step("2) bob asks again → HIT")
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    _step("3) LEAK TEST: alice asks the same finance question → DENIED")
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    _step("4) LEAK TEST: alice asks a paraphrase → DENIED")
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    _step("5) unknown user → DENIED")
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    _step("6) alice asks about the OpenAI docs (shared source) → MISS, then HIT")
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    _step("7) PER-SOURCE CACHE: bob asks the same OpenAI question → HIT on alice's answer")
    r = secure_agentic_rag_cached("bob", q_doc, cache)
    assert r["status"] == "HIT", f"expected HIT from the shared OPENAI_QUERY partition, got {r['status']}"

    _step("8) time-sensitive question asked twice → MISS both times, nothing stored")
    before = cache.sizes()
    for _ in range(2):
        r = secure_agentic_rag_cached("alice", q_live, cache)
        assert r["status"] == "MISS" and not r["cached"], f"time-sensitive answer was cached: {r}"
    assert cache.sizes() == before, "time-sensitive question changed the cache"

    print(f"\nCached answers per source: {cache.sizes()}")
    print("✅ All checks passed — cache is fast and does not leak across roles.")


run_self_check()

Loading the cache embedding model (once)...



1) bob asks a finance question for the first time → MISS
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Uber's 2021 revenue is reported in its annual filing

❌ Cache MISS — running the 10K_DOCUMENT_QUERY pipeline...
💾 Stored in the 10K_DOCUMENT_QUERY partition.
Status: MISS  (2.96s)

🤖 BOT RESPONSE:
Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]


2) bob asks again → HIT
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Requests financial data from Uber's annual report

✅ Cache HIT in the 10K_DOCUMENT_QUERY partition (similarity 1.000)
Status: HIT  (1.13s)

🤖 BOT RESPONSE:
Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]


3) LEAK TEST: alice asks the same finance question → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?

📍 Select